In [1]:
import pandas as pd
import numpy as np
import os
import glob # 用于查找文件路径

# 请将这里替换为你存放所有股票数据的根目录
# 例如：'F:/self_quant/data/data/'
# 假设你的目录结构是：
# - F:/self_quant/data/data/
#   - 退市票利润/
#   - 退市票现金流量/
#   - ...
DATA_PATH = 'F:/self_quant/data/data/'

# 为了演示，我们先使用你提供的示例股票代码
STOCK_CODE = 'sz.002231'


In [2]:
# 定义三大表的列名（这里只选取了部分重要指标）
# 利润表列名
income_columns = [
    '报告日', '净利润', '营业总收入', '营业总成本', '营业成本',
    '营业利润', '利润总额', '所得税费用', '归属于母公司所有者的净利润'
]

# 资产负债表列名
balance_columns = [
    '报告日', '货币资金', '应收账款', '存货', '流动资产合计',
    '非流动资产合计', '资产总计', '流动负债合计', '非流动负债合计',
    '负债合计', '所有者权益(或股东权益)合计', '归属于母公司所有者权益(或股东权益)合计'
]

# 现金流量表列名
cashflow_columns = [
    '报告日', '经营活动产生的现金流量净额', '投资活动产生的现金流量净额',
    '筹资活动产生的现金流量净额', '现金及现金等价物净增加额'
]

# 用于加载和初步清洗财务报表
def load_financial_statement(file_path, columns):
    """
    加载单个财务报表文件，并增加【公告日期】
    :param file_path: 文件路径
    :param columns: 我们关心的列名列表
    :return: 清洗后的DataFrame
    """
    # 1. 正常读取数据，让 pandas 自动读取第一行作为表头(header=0)
    # 如果你的 CSV 第一行真的是需要跳过的垃圾信息，第二行才是列名，请保留 skiprows=1，去掉 header=None
    df_full = pd.read_csv(file_path, encoding='gbk')

    # 2. 找出需要提取且在实际数据中存在的列，防止某些列名在 CSV 中不存在导致报错
    valid_columns = [col for col in columns if col in df_full.columns]

    # 按列名精确提取所需数据
    df = df_full[valid_columns].copy()


    # --- 数据清洗 ---
    # 1. 将'报告日'转换为日期格式
    df['报告日'] = pd.to_datetime(df['报告日'], format='%Y%m%d')

    return df

# 重新加载示例股票的三大报表
profit_df = load_financial_statement(os.path.join(DATA_PATH, '退市票利润/sz002231_ST奥维_利润表.csv'), income_columns)
balance_df = load_financial_statement(os.path.join(DATA_PATH, '退市票资产负债/sz002231_ST奥维_资产负债表.csv'), balance_columns)
cashflow_df = load_financial_statement(os.path.join(DATA_PATH, '退市票现金流量/sz002231_ST奥维_现金流量表.csv'), cashflow_columns)

print("利润表")
profit_df.head(30)

利润表 (前5条)，注意新增了'公告日期'列


,报告日,净利润,营业总收入,营业总成本,营业成本,营业利润,利润总额,所得税费用,归属于母公司所有者的净利润
0,2025-09-30,-2.375861e+08,3.400250e+07,7.192178e+07,4.480586e+07,-2.011160e+08,-2.377638e+08,-177672.16,-1.876372e+08
1,2025-06-30,-1.131487e+08,2.347618e+07,5.431016e+07,3.321256e+07,-8.935789e+07,-1.132271e+08,-78345.71,-8.906687e+07
2,2025-03-31,-7.525846e+06,2.249772e+07,3.213693e+07,2.284278e+07,-7.972651e+06,-8.061474e+06,-535628.20,-6.339408e+06
3,2024-12-31,-4.820824e+07,2.912910e+08,3.186087e+08,2.763075e+08,-4.382937e+07,-4.158385e+07,6624385.88,-4.611472e+07
4,2024-09-30,-1.801368e+07,2.616635e+08,2.827403e+08,2.526771e+08,-2.100177e+07,-1.842856e+07,-414887.94,-1.724608e+07
5,2024-06-30,-6.522695e+06,2.097432e+08,2.226824e+08,2.025690e+08,-8.558511e+06,-6.885148e+06,-362453.43,-5.793644e+06
6,2024-03-31,-4.025451e+06,1.018921e+08,1.069245e+08,9.761450e+07,-5.764781e+06,-4.410929e+06,-385477.86,-4.025451e+06
7,2023-12-31,-3.420479e+07,1.724311e+08,1.986314e+08,1.500826e+08,-3.112925e+07,-3.024677e+07,3958019.73,-3.420479e+07
8,2023-09-30,-2.022847e+06,9.944045e+07,1.140562e+08,8.029969e+07,-1.061957e+06,-1.110035e+06,912811.92,-2.022847e+06
9,2023-06-30,1.186891e+07,7.851513e+07,8.571482e+07,6.434245e+07,1.203190e+07,1.186891e+07,NaN,1.186891e+07


In [3]:
balance_df.head(5)

,报告日,货币资金,应收账款,存货,流动资产合计,非流动资产合计,资产总计,流动负债合计,非流动负债合计,负债合计,所有者权益(或股东权益)合计,公告日期
0,2025-09-30,87809647.36,20429192.37,8.907295e+07,2.073783e+08,38858799.84,2.462371e+08,1.491790e+08,9416409.26,1.585954e+08,8.764174e+07,2025-10-31
1,2025-06-30,69748937.81,36598423.39,8.963635e+07,3.261156e+08,40051111.17,3.661667e+08,1.535837e+08,503862.50,1.540876e+08,2.120792e+08,2025-08-28
2,2025-03-31,20910343.01,50040378.61,1.251003e+08,4.487562e+08,44549486.11,4.933057e+08,1.734739e+08,2129737.00,1.756037e+08,3.177020e+08,2025-04-29
3,2024-12-31,64801339.41,50387434.61,1.461017e+08,4.675232e+08,45036644.58,5.125598e+08,1.849422e+08,2389748.09,1.873319e+08,3.252279e+08,2025-10-31
4,2024-09-30,46153453.50,66896033.50,2.010901e+08,5.145348e+08,52755994.05,5.672908e+08,2.098126e+08,2055769.33,2.118684e+08,3.554224e+08,2025-04-29


In [4]:
cashflow_df.head(5)

,报告日,经营活动产生的现金流量净额,投资活动产生的现金流量净额,筹资活动产生的现金流量净额,现金及现金等价物净增加额,公告日期
0,2025-09-30,-30974688.38,1176518.05,-459417.62,-30256090.95,2025-10-31
1,2025-06-30,-47761463.38,-121481.95,-426692.62,-48308140.95,2025-08-28
2,2025-03-31,-41676326.18,-285506.83,-229163.39,-42190996.40,2025-04-29
3,2024-12-31,-71527640.76,25247477.65,49703640.61,3440936.09,2025-04-29
4,2024-09-30,-86881238.88,25322630.11,48913645.81,-12608074.35,2025-10-31
